<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase12_valores_propios_pca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import yfinance as yf

# 1. Comprobar los resultados a mano
B = np.array([[3, 1], [1, 3]])
valores, vectores = np.linalg.eigh(B)
print("Valores propios de B:", valores)

# 2. PCA con tres pares de divisas
datos = yf.download(["MXN=X", "JPY=X", "EURUSD=X"],
                    start="2020-10-07", end="2026-10-22", progress=False)["Close"]
datos["EURUSD=X"] = 1 / datos["EURUSD=X"]
rend = datos.pct_change().dropna()

corr = rend.corr().values
valores, vectores = np.linalg.eigh(corr)
orden = np.argsort(valores)[::-1]
valores = valores[orden]
vectores = vectores[:, orden]

print("Varianza explicada por componente:", (valores / valores.sum()).round(3))
print("Primer componente:")
for nombre, peso in zip(rend.columns, vectores[:, 0]):
    print(" ", nombre, round(float(peso), 2))

Valores propios de B: [2. 4.]


/tmp/ipykernel_1849/724962724.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  datos = yf.download(["MXN=X", "JPY=X", "EURUSD=X"],


Varianza explicada por componente: [0.569 0.288 0.143]
Primer componente:
  EURUSD=X -0.67
  JPY=X -0.56
  MXN=X -0.49


In [ ]:
for i in range(3):
    print("Componente", i + 1, "| varianza explicada:", round(float(valores[i] / valores.sum()), 3))
    for nombre, peso in zip(rend.columns, vectores[:, i]):
        print("  ", nombre, round(float(peso), 2))

Componente 1 | varianza explicada: 0.569
   EURUSD=X -0.67
   JPY=X -0.56
   MXN=X -0.49
Componente 2 | varianza explicada: 0.288
   EURUSD=X -0.05
   JPY=X -0.63
   MXN=X 0.78
Componente 3 | varianza explicada: 0.143
   EURUSD=X -0.74
   JPY=X 0.54
   MXN=X 0.39


In [3]:
z = (rend - rend.mean()) / rend.std()
puntajes = z.values @ vectores

promedio = z.mean(axis=1)
dif_peso_yen = z["MXN=X"] - z["JPY=X"]

print("Componente 1 contra promedio de los tres:", round(float(np.corrcoef(puntajes[:, 0], promedio)[0, 1]), 2))
print("Componente 2 contra peso menos yen:", round(float(np.corrcoef(puntajes[:, 1], dif_peso_yen)[0, 1]), 2))

Componente 1 contra promedio de los tres: -1.0
Componente 2 contra peso menos yen: 0.99


In [4]:
import pandas as pd

c2 = pd.Series(puntajes[:, 1], index=rend.index)
extremos = c2.abs().sort_values(ascending=False).head(8)
print(pd.DataFrame({"fecha": extremos.index.date, "valor": c2.loc[extremos.index].round(2).values}))

        fecha  valor
0  2024-08-05   6.44
1  2024-06-04   5.77
2  2023-03-14   5.34
3  2025-04-07   4.54
4  2025-04-11   4.47
5  2025-02-04  -4.07
6  2024-08-09  -3.93
7  2025-04-10  -3.90


In [5]:
fechas = c2.abs().sort_values(ascending=False).head(8).index
print(pd.DataFrame({
    "MXN": z.loc[fechas, "MXN=X"].round(2),
    "JPY": z.loc[fechas, "JPY=X"].round(2),
    "EUR": z.loc[fechas, "EURUSD=X"].round(2),
    "componente 2": c2.loc[fechas].round(2),
}))

             MXN   JPY   EUR  componente 2
Date                                      
2024-08-05  4.72 -4.23 -2.49          6.44
2024-06-04  6.33 -1.26 -1.10          5.77
2023-03-14  4.85 -2.43 -0.86          5.34
2025-04-07  5.11 -0.99  1.18          4.54
2025-04-11  1.90 -4.32 -5.98          4.47
2025-02-04 -5.44 -0.14 -1.70         -4.07
2024-08-09 -3.60  1.78  0.24         -3.93
2025-04-10 -3.54  1.79  0.56         -3.90
